### Calculate coefficient of variation as per Glüer et al. (1995)

Author: Simone Poncioni, MSB, ARTORG Center for Biomedical Engineering Research, University of Bern, Switzerland

Date: 07.2024

Update: 29.04.2025 for evaluating PE as per Schenk et al. (2020)

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from collections import Counter, defaultdict

import matplotlib.pyplot as plt

In [4]:
def calculate_cv_gluer(infile: pd.DataFrame, outfile: dict, variable: str):
    """Function to calculate the coefficient of variation as per Glüer et al. 1995

    Args:
        infile (pd.DataFrame): DataFrame containing the imported yaml file with the simulation grouping information
        outfile (dict): simulation results file
        variable (str): variable to analyse (e.g. 'Stiffness', 'yield_force', etc.)
        
    Returns:
        pd.DataFrame: DataFrame with precision error results
    """
    def dofs(results):
        '''
        Equation (7), calculating the degrees of freedom df_j of the measurements in the individuals
        '''
        # Create a counter for the number of measurements per patient
        counter = Counter()
        for key in results.keys():
            # Extract patient ID the same way as in precision_error function
            patient_id = key.split('_meas')[0]
            counter[patient_id] += 1
        
        # Calculate degrees of freedom
        degrees_of_freedom = sum(count - 1 for count in counter.values())
        return degrees_of_freedom

    def precision_error(results):
        '''
        Equation (6), calculating the generic standard deviation sd of the measurements
        '''
        values_sorted = defaultdict(list)
        for key, value in results.items():
            # Extract just the patient ID part without the measurement number
            patient_id = key.split('_meas')[0]
            values_sorted[patient_id].append(value[1])
    
        
        # Filter out participants with only one measurement
        valid_participants = {k: v for k, v in values_sorted.items() if len(v) >= 2}
        
        if not valid_participants:
            return None, values_sorted
            
        sum_tot = 0
        total_elements = 0
        df = 0

        # Calculating the internal mean, sum of squared differences, and degrees of freedom
        for key, values in valid_participants.items():
            internal_mean = sum(values) / len(values)
            total_elements += len(values)
            df += len(values) - 1  # Adding to degrees of freedom
            
            for xij in values:
                sum_tot += (xij - internal_mean) ** 2

        # Check if we have enough degrees of freedom
        if df == 0:
            return None, values_sorted

        # Dividing by degrees of freedom
        variance = sum_tot / df
        sd = variance ** 0.5

        return sd, values_sorted

    def coefficient_of_variation(sd, values_sorted):
        '''
        Equation (5), calculating the coefficient of variation on a percentage basis
        '''
        HUNDRED = 100
        sum_means = 0
        # Only consider participants with multiple measurements
        valid_participants = {k: v for k, v in values_sorted.items() if len(v) >= 2}
        m = len(valid_participants)
        
        if m == 0:
            return None, None

        for _, values in valid_participants.items():
            internal_mean = sum(values) / len(values)
            sum_means += internal_mean
        
        mean_of_means = sum_means / m
        cv_sd = (sd / mean_of_means) * HUNDRED
        return cv_sd, mean_of_means
    
    # Get the measurements for the specific variable from outfile
    results = {}
    for key, data in outfile.items():
        if variable in data:
            # Format the value as (patient_id/measurement_id, value)
            patient_id = key
            measurement_id = data.get('MeasNo', 'unknown')
            path = f"patient/{patient_id}/measurement/{measurement_id}"
            results[key] = (path, data[variable])
    
    # Handle the case with no or insufficient data
    if not results:
        return pd.DataFrame({
            'Variable': [variable],
            'Mean': [None],
            'SD': [None],
            'CV (%)': [None],
            'DF': [0],
            'n': [0],
        })
    
    # Calculate precision error and CV
    sd, values_sorted = precision_error(results)
    
    # Handle case where precision error calculation wasn't possible
    if sd is None:
        return pd.DataFrame({
            'Variable': [variable],
            'Mean': [None],
            'SD': [None],
            'CV (%)': [None],
            'DF': [0],
            'n': [len(values_sorted)],
        })
    
    cv, mean_value = coefficient_of_variation(sd, values_sorted)
    df = dofs(results)
    
    # Create a DataFrame with the results
    result_df = pd.DataFrame({
        'Variable': [variable],
        'Mean': [mean_value],
        'SD': [sd],
        'CV (%)': [cv],
        'DF': [df],
        'n': [len(values_sorted)],
    })
    
    return result_df

In [5]:
def load_and_preprocess_data(file_path):
    """
    Load and preprocess HR-pQCT data from CSV file
    
    Args:
        file_path (str): Path to the HR-pQCT data CSV file
        
    Returns:
        pd.DataFrame: Processed DataFrame
    """
    # Load data
    df = pd.read_csv(file_path)
    return df

def extract_property_names(df):
    """
    Extract unique property names from column names by removing numeric suffixes
    """
    properties = set()
    for col in df.columns:
        # Check if column ends with a digit 1-3 and has at least one character before it
        if len(col) > 1 and col[-1] in '123' and col[-2] != '.':
            base_property = col[:-1]  # Remove the numeric suffix
            properties.add(base_property)
    return sorted(list(properties))

def prepare_data_for_analysis(df, property_list):
    """
    Prepare data in the format needed for precision error calculation
    
    Args:
        df (pd.DataFrame): Input DataFrame with HR-pQCT data
        property_list (list): List of properties to analyze
        
    Returns:
        dict: Dictionary formatted for analysis
    """
    # Create a dictionary to hold the output data
    outfile = {}
    
    # Process each row (participant)
    for idx, row in df.iterrows():
        patient_id = row['PatNo'] if 'PatNo' in row else f"patient_{idx}"
        
        # Process each measurement (1, 2, 3)
        for meas_num in range(1, 4):
            # Check if this measurement exists (by checking if any property has this measurement)
            if any(f"{prop}{meas_num}" in df.columns and pd.notna(row[f"{prop}{meas_num}"]) 
                   for prop in property_list):
                
                # Create a unique key for this patient and measurement
                key = f"{patient_id}_meas{meas_num}"
                outfile[key] = {'MeasNo': meas_num}
                
                # Add all properties for this measurement
                for prop in property_list:
                    col_name = f"{prop}{meas_num}"
                    if col_name in df.columns and pd.notna(row[col_name]):
                        outfile[key][prop] = row[col_name]
    
    return outfile

def analyze_precision_error(df, properties_to_analyze=None):
    """
    Analyze precision error for all properties in the dataset
    
    Args:
        df (pd.DataFrame): Input DataFrame with HR-pQCT data
        properties_to_analyze (list, optional): List of specific properties to analyze
                                              If None, all properties will be analyzed
    
    Returns:
        pd.DataFrame: DataFrame with precision error results for all properties
    """
    # Extract all property names
    all_properties = extract_property_names(df)
    
    # Filter properties if a specific list is provided
    if properties_to_analyze is not None:
        properties = [p for p in all_properties if p in properties_to_analyze]
    else:
        properties = all_properties
    
    # Prepare data for analysis
    outfile = prepare_data_for_analysis(df, properties)
    
    # Create a dummy infile (not actually used in the calculations)
    infile = pd.DataFrame()
    
    # Calculate precision error for each property
    results_df = pd.DataFrame()
    print(properties)
    for prop in properties:
        prop_result = calculate_cv_gluer(infile, outfile, prop)
        results_df = pd.concat([results_df, prop_result], ignore_index=True)
    
    return results_df

def pivot_dataframe(df):
    """
    Pivot the results dataframe for better presentation
    """
    pivot_df = df.set_index('Variable')
    return pivot_df

In [6]:
def analyze_precision_error_by_site(df, properties_to_analyze=None):
    """
    Analyze precision error for radius and tibia separately
    
    Args:
        df (pd.DataFrame): Input DataFrame with HR-pQCT data
        properties_to_analyze (list, optional): List of base property names to analyze
                                             If None, all properties will be analyzed
    
    Returns:
        tuple: (radius_results, tibia_results) DataFrames with precision error results
    """
    # Extract all property names
    all_properties = extract_property_names(df)
    
    # Separate radius and tibia properties
    radius_properties = [p for p in all_properties if 'Radius' in p or 'Rad_' in p or 'R_' in p]
    tibia_properties = [p for p in all_properties if 'Tibia' in p or 'Tib_' in p or 'T_' in p]
    
    # If the column names don't contain 'Radius' or 'Tibia' prefixes,
    # check if there are anatomical site indicators in other columns
    if not radius_properties and not tibia_properties:
        if 'Site' in df.columns:
            # Create separate dataframes for radius and tibia
            radius_df = df[df['Site'].str.contains('Radius|R|rad', case=False, na=False)].copy()
            tibia_df = df[df['Site'].str.contains('Tibia|T|tib', case=False, na=False)].copy()
            
            # Analyze precision error for each site
            radius_results = analyze_precision_error(radius_df, properties_to_analyze)
            radius_results['Site'] = 'Radius'
            
            tibia_results = analyze_precision_error(tibia_df, properties_to_analyze)
            tibia_results['Site'] = 'Tibia'
            
            # Add missing return statement here
            return radius_results, tibia_results
        else:
            # If we can't determine the site, just analyze all properties
            # and let the user decide which are radius vs tibia
            print("Unable to automatically determine radius and tibia properties.")
            print("Please specify the prefix or pattern for radius and tibia properties.")
            
            # For this implementation, we'll assume properties are repeated for each site
            # with site names in separate columns (common in HR-pQCT datasets)
            radius_results = analyze_precision_error(df, properties_to_analyze)
            radius_results['Site'] = 'Radius'
            
            tibia_results = analyze_precision_error(df, properties_to_analyze)
            tibia_results['Site'] = 'Tibia'
            
            return radius_results, tibia_results
    else:
        # Analyze radius and tibia properties separately
        if properties_to_analyze:
            radius_props = [p for p in radius_properties if any(base in p for base in properties_to_analyze)]
            tibia_props = [p for p in tibia_properties if any(base in p for base in properties_to_analyze)]
        else:
            radius_props = radius_properties
            tibia_props = tibia_properties
        
        radius_results = analyze_precision_error(df, radius_props)
        radius_results['Site'] = 'Radius'
        
        tibia_results = analyze_precision_error(df, tibia_props)
        tibia_results['Site'] = 'Tibia'
        
        return radius_results, tibia_results

def analyze_data_availability(df, properties_to_analyze):
    """
    Analyze how many measurements are available for each property and patient
    
    Args:
        df (pd.DataFrame): Input DataFrame with HR-pQCT data
        properties_to_analyze (list): List of properties to analyze
        
    Returns:
        pd.DataFrame: Summary of data availability
    """
    availability = {}
    
    for prop in properties_to_analyze:
        radius_cols = [col for col in df.columns if prop in col and ('Radius' in col or 'R_' in col or 'Rad_' in col)]
        tibia_cols = [col for col in df.columns if prop in col and ('Tibia' in col or 'T_' in col or 'Tib_' in col)]
        
        # If no site-specific columns found, look for columns with just the property name and a measurement number
        if not radius_cols and not tibia_cols:
            prop_cols = [col for col in df.columns if col.startswith(prop) and col[-1] in '123']
            if prop_cols:
                # Count non-null values for each measurement number
                counts = {}
                for i in range(1, 4):
                    count = sum(1 for col in prop_cols if col.endswith(str(i)) and df[col].notna().sum() > 0)
                    counts[f'Measurement {i}'] = count
                
                availability[prop] = counts
        else:
            # Count non-null values for radius and tibia
            radius_counts = {}
            tibia_counts = {}
            
            for i in range(1, 4):
                r_cols = [col for col in radius_cols if col.endswith(str(i))]
                t_cols = [col for col in tibia_cols if col.endswith(str(i))]
                
                radius_counts[f'Measurement {i}'] = sum(df[col].notna().sum() for col in r_cols)
                tibia_counts[f'Measurement {i}'] = sum(df[col].notna().sum() for col in t_cols)
            
            availability[f'Radius: {prop}'] = radius_counts
            availability[f'Tibia: {prop}'] = tibia_counts
    
    # Convert to DataFrame for easier viewing
    availability_df = pd.DataFrame(availability)
    return availability_df.T  # Transpose for better readability

def plot_cv_results(radius_results, tibia_results):
    """
    Create visualizations of the CV% results
    
    Args:
        radius_results (pd.DataFrame): DataFrame with radius precision error results
        tibia_results (pd.DataFrame): DataFrame with tibia precision error results
    """
    # Filter out rows with missing CV values
    radius_valid = radius_results[radius_results['CV (%)'].notna()].copy()
    tibia_valid = tibia_results[tibia_results['CV (%)'].notna()].copy()
    
    # Check if there's data to plot
    if radius_valid.empty and tibia_valid.empty:
        print("No valid CV% values to plot.")
        return
    
    # Set up the figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 10))
    fig.suptitle('Coefficient of Variation (CV%) by Site', fontsize=16)
    
    # Sort by CV% for better visualization
    radius_valid = radius_valid.sort_values('CV (%)')
    tibia_valid = tibia_valid.sort_values('CV (%)')
    
    # Plot radius data
    if not radius_valid.empty:
        bars1 = ax1.barh(radius_valid['Variable'], radius_valid['CV (%)'], color='skyblue')
        ax1.set_title('Radius')
        ax1.set_xlabel('CV (%)')
        ax1.grid(axis='x', linestyle='--', alpha=0.7)
        
        # Add value labels
        for bar in bars1:
            width = bar.get_width()
            ax1.text(width + 0.1, bar.get_y() + bar.get_height()/2, f'{width:.2f}%', 
                    ha='left', va='center', fontsize=8)
    
    # Plot tibia data
    if not tibia_valid.empty:
        bars2 = ax2.barh(tibia_valid['Variable'], tibia_valid['CV (%)'], color='lightgreen')
        ax2.set_title('Tibia')
        ax2.set_xlabel('CV (%)')
        ax2.grid(axis='x', linestyle='--', alpha=0.7)
        
        # Add value labels
        for bar in bars2:
            width = bar.get_width()
            ax2.text(width + 0.1, bar.get_y() + bar.get_height()/2, f'{width:.2f}%', 
                    ha='left', va='center', fontsize=8)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

def load_and_preprocess_excel_data(file_path):
    """
    Load and preprocess HR-pQCT data from Excel file
    
    Args:
        file_path (str): Path to the HR-pQCT data Excel file
        
    Returns:
        pd.DataFrame: Processed DataFrame
    """
    # Load data
    df = pd.read_excel(file_path)
    return df

In [7]:
# Denis's repro data
excel_path = '/home/simoneponcioni/Documents/01_PHD/03_Methods/HR-pQCT_database/03_EVALUATION/04_rate-change/precision_error/denis/Repro_UPAT_TOTAL_Denis_Cleaned_004_incl_nm_003.xlsx'

df = load_and_preprocess_excel_data(excel_path)

# Define specific properties to analyze (common HR-pQCT parameters)
properties_to_analyze = [
    'Tot.vBMD', 'Ct.vBMD', 'Tb.BV.TV',
    'F.load', 'Stiffness', 
    'Ct.Th', 'Ct.Po',
    'Tb.vBMD',
    'Tb.N', 'Tb.Sp', 'Tb.Th',
]

# Calculate precision error and CV for radius and tibia separately
radius_results, tibia_results = analyze_precision_error_by_site(df, properties_to_analyze)


# Create pivot tables for easier viewing
radius_pivot = pivot_dataframe(radius_results)
tibia_pivot = pivot_dataframe(tibia_results)

print("\nPivot Table - RADIUS:")
print(radius_pivot)

print("\nPivot Table - TIBIA:")
print(tibia_pivot)

# # Create visualizations
# plot_cv_results(radius_results, tibia_results)

# # Save results to CSV
output_dir = Path('/home/simoneponcioni/Documents/01_PHD/03_Methods/HR-pQCT_database/03_EVALUATION/04_rate-change/precision_error/results')
output_dir.mkdir(exist_ok=True)

radius_results.to_csv(output_dir / 'radius_precision_error.csv', index=False)
tibia_results.to_csv(output_dir / 'tibia_precision_error.csv', index=False)

# Create a combined results file
combined_results = pd.concat([radius_results, tibia_results], ignore_index=True)
combined_results.to_csv(output_dir / 'combined_precision_error.csv', index=False)

print(f"\nResults saved to {output_dir}")

['Ct.Po', 'Ct.Th', 'Ct.vBMD', 'F.load', 'Stiffness', 'Tb.BV.TV', 'Tb.N', 'Tb.Sp', 'Tb.Th', 'Tb.vBMD', 'Tot.vBMD']
['Ct.Po', 'Ct.Th', 'Ct.vBMD', 'F.load', 'Stiffness', 'Tb.BV.TV', 'Tb.N', 'Tb.Sp', 'Tb.Th', 'Tb.vBMD', 'Tot.vBMD']

Pivot Table - RADIUS:
                   Mean           SD     CV (%)  DF   n    Site
Variable                                                       
Ct.Po          0.009475     0.001067  11.261839  66  34  Radius
Ct.Th          1.153230     0.021256   1.843146  66  34  Radius
Ct.vBMD      882.764706    11.010420   1.247266  66  34  Radius
F.load     -7030.857387   527.446947  -7.501887  66  34  Radius
Stiffness  40059.658824  1691.955125   4.223588  66  34  Radius
Tb.BV.TV       0.257838     0.009539   3.699502  66  34  Radius
Tb.N           1.610431     0.045084   2.799497  66  34  Radius
Tb.Sp          0.586127     0.016798   2.866001  66  34  Radius
Tb.Th          0.233176     0.004570   1.959839  66  34  Radius
Tb.vBMD      178.965196     5.930820   3.3139

In [8]:
def rel_cortical_thickness(cortical_thickness_mm, total_area):
    return cortical_thickness_mm / np.sqrt(total_area / np.pi)

In [12]:
# Estimate relative cortical thickness from Denis's data
input_path = Path("/home/simoneponcioni/Documents/01_PHD/03_Methods/HR-pQCT_database/03_EVALUATION/04_rate-change/precision_error/Repro_UPAT_TOTAL_Denis_Cleaned_004_incl_nm_003_includedFRAX.csv")

df_denis = pd.read_csv(input_path)

df_denis["Tot.Ar1"] = df_denis['Ct.Ar1'] + df_denis['Tb.Ar1']
df_denis["Tot.Ar2"] = df_denis['Ct.Ar2'] + df_denis['Tb.Ar2']
df_denis["Tot.Ar3"] = df_denis['Ct.Ar3'] + df_denis['Tb.Ar3']
df_denis["relctth1"] = rel_cortical_thickness(df_denis['Ct.Th1'], df_denis['Tot.Ar1'])
df_denis["relctth2"] = rel_cortical_thickness(df_denis['Ct.Th2'], df_denis['Tot.Ar2'])
df_denis["relctth3"] = rel_cortical_thickness(df_denis['Ct.Th3'], df_denis['Tot.Ar3'])

# Define specific properties to analyze (common HR-pQCT parameters)
properties_to_analyze = ['relctth', 'Tot.Ar', 'Ct.Ar', 'Tb.Ar']

# Calculate precision error and CV for radius and tibia separately
radius_results, tibia_results = analyze_precision_error_by_site(df_denis, properties_to_analyze)


# Create pivot tables for easier viewing
radius_pivot = pivot_dataframe(radius_results)
tibia_pivot = pivot_dataframe(tibia_results)

print("\nPivot Table - RADIUS:")
print(radius_pivot)

print("\nPivot Table - TIBIA:")
print(tibia_pivot)

['Ct.Ar', 'Tb.Ar', 'Tot.Ar', 'relctth']
['Ct.Ar', 'Tb.Ar', 'Tot.Ar', 'relctth']

Pivot Table - RADIUS:
                Mean         SD     CV (%)  DF   n    Site
Variable                                                  
Ct.Ar      76.210784   3.844214   5.044186  66  34  Radius
Tb.Ar     424.182353  50.893322  11.997982  66  34  Radius
Tot.Ar    500.393137  54.580778  10.907579  66  34  Radius
relctth     0.102400   0.003287   3.209828  66  34  Radius

Pivot Table - TIBIA:
                 Mean        SD    CV (%)  DF   n   Site
Variable                                                
Ct.Ar      113.464103  1.285625  1.133067  73  39  Tibia
Tb.Ar     1003.712821  3.438149  0.342543  73  39  Tibia
Tot.Ar    1117.176923  2.977580  0.266527  73  39  Tibia
relctth      0.064835  0.000852  1.314712  73  39  Tibia


In [10]:
# Novel hFE results (Poncioni et al. 2025) for apparent stress, trabecular anisotropy, and relative cortical thickness

res_path = Path('/home/simoneponcioni/Documents/01_PHD/03_Methods/HR-pQCT_database/03_EVALUATION/08_hfe-evaluation/apparent_stress/001_repro_paper_data_summary.app_yield_stress.csv')

res_df = pd.read_csv(res_path)

# separate radius and tibia results based on 'height column: > 20 is tibia, <= 20 is radius
radius_df = res_df[res_df['height'] <= 20].copy()
tibia_df = res_df[res_df['height'] > 20].copy()


In [11]:
# Reproducibility calculated from Michi's filtered data

# 1. Import Michi's data
# 2. Import UCT_LIST
# 3. Merge the two dataframes
# 4. Calculate the CV as per Gluer et al. (1995) on the new columns